About

In [16]:
# This script is used for data preprocessing and similarity analysis.
# Author: Srihariharasudhan Balakannan
# Date: [2025-02-16]

___

Install pyspark and create spark session

In [20]:
# install dependancies
!pip install pyspark > /dev/null 2>&1

In [13]:
# imports
from pyspark.sql import SparkSession

In [14]:
spark = SparkSession.builder.appName("SparkJobFit").getOrCreate()

In [15]:
spark

___

Get necessary data

In [17]:
# install necessary packages
!pip install PyMuPDF > /dev/null 2>&1

In [18]:
# necessary imports
import pandas as pd
from google.colab import drive
import fitz  # PyMuPDF

In [19]:
# function to parse data from pdf file and return in text format
def parse_pdf(file_path) -> str:
  txt = ""
  doc = fitz.open(file_path)
  for page in doc:
    txt += page.get_text("text") + "\n"
  return txt

In [21]:
# parse sample resume data in text format
drive.mount('/content/drive')
file_path     = '/content/drive/My Drive/Colab Notebooks/SparkJobFit/data-engineer-resume-example.pdf'
resume_data   = parse_pdf(file_path)

Mounted at /content/drive


In [23]:
print(resume_data)

ALAN SUSA
Data Engineer
alansusa@email.com
(123) 456-7890
New York, NY
LinkedIn
WORK EXPERIENCE
Data Engineer
Consumer Reports
May 2018 - current
New York, NY
Led the migration from Oracle to Redshift using Amazon Athena
and S3, resulting in an annual cost savings of $678,000 and an
increase in performance of 14%
Designed and implemented a real-time data pipeline to process
semi-structured data by integrating 150 million raw records
from 30+ data sources using Kafka and PySpark
Designed the data pipeline architecture for a new product that
quickly scaled from 0 to 125,000 daily active users
Studied and revamped data dictionaries to include a more
robust history for developing consistency across domain
Data Engineer
Guardian Life Insurance Company
August 2016 - May 2018
New York, NY
Maintained data pipeline up-time of 99.8% while ingesting
streaming and transactional data across 8 different primary
data sources using Spark, Redshift, S3, and Python
Automated ETL processes across billion

In [24]:
# sample job description
job_description = """About the job
Position Summary...

What you'll do...

About Team:

Our Team focuses on managing and delivering world-class data assets, including creating and maintaining data standards, driving policy compliance, creating partnerships, and developing pipelines and self-service tools. DCA > SC Stores Team is one of the critical team within End to End Fulfilment area of Data and Customer Analytics handling Supply chain Stores metrics which enables data driven decision making.

What you'll do:

Design, build, test and deploy cutting edge solutions at scale, impacting multi-billion-dollar business.
Work closely with product owner and technical lead and play a major role in the overall delivery of the assigned project/enhancements.
 Interact with Walmart engineering teams across geographies to leverage expertise and contribute to the tech community.
Provide business insights while leveraging internal tools and systems, databases and industry data.
Drive the success of the implementation by applying technical skills, to design and build enhanced processes and technical solutions in support of strategic initiatives.

What you'll bring:

4-6+ years of Data Engineering experience in Big Data technologies like Distributed computing frameworks
Experience in deploying solutions on any of cloud platforms (prefer GCP)
Working knowledge of SQL /No-SQL and database technologies & big data skills Hadoop, SPARK, Scala, PySpark, SQL, Query Optimization
Work exposure on Agile methodologies and DevOps would be added advantage
Experience in Data Model design & CI/CD tools.
Experience with Orchestration\Scheduling tools like Airflow/ Automic etc.
Good Knowledge to Streaming data use-cases via Kafka or structured streaming etc
Learn & Research on the go and work on both new requests/projects as well as support L2/L3 in production
Experience &expertise in Data processing and Data manipulation skills like Data warehousing concepts, SCD types etc
Exceptional communication and interpersonal skills - including negotiation, facilitation, and consensus building skills; ability to influence and persuade, without direct control

About Walmart Global Tech

From entry-level to executive positions, Walmart provides limitless opportunities for growth, and career development. Walmart started small, with a single discount store and the simple philosophy of selling more for less. Today, we are a growing technology-enabled company founded on the same values as our first store. We establish clear expectations, empower associates to manage their work, and hold ourselves and one another to a high standard. Walmart's scale enables us to have an. No other company has the reach of Walmart, with 2.3 million associates worldwide and over 230 million weekly customers. Walmart is reshaping retail by investing in an expanding workforce. While technology is at the heart of our digital transformation, people are the reason we succeed and the force behind our innovations. We train our team in the skillsets of the future and bring in experts like you to help us grow.

Flexible, hybrid work

We use a hybrid way of working with primary in office presence coupled with an optimal mix of virtual presence. We use our campuses to collaborate and be together in person, as business needs require and for development and networking opportunities. This approach helps us make quicker decisions, remove location barriers across our global team, be more flexible in our personal lives.

Benefits

Beyond our great compensation package, you can receive incentive awards for your performance. Other great perks include a host of best-in-class benefits maternity and parental leave, PTO, health benefits, and much more.

Equal Opportunity Employer:

Walmart, Inc. is an Equal Opportunity Employer – By Choice. We believe we are best equipped to help our associates, customers and the communities we serve live better when we really know them. That means understanding, respecting and valuing diversity- unique styles, experiences, identities, ideas and opinions – while being inclusive of all people

Minimum Qualifications...

Outlined below are the required minimum qualifications for this position. If none are listed, there are no minimum qualifications.

Minimum Qualifications:Option 1: Bachelor's degree in Computer Science and 2 years' experience in software engineering or related field. Option 2: 4 years' experience in software engineering or related field. Option 3: Master's degree in Computer Science.

Preferred Qualifications...

Outlined below are the optional preferred qualifications for this position. If none are listed, there are no preferred qualifications.
"""

In [25]:
print(job_description)

About the job
Position Summary...

What you'll do...

About Team:

Our Team focuses on managing and delivering world-class data assets, including creating and maintaining data standards, driving policy compliance, creating partnerships, and developing pipelines and self-service tools. DCA > SC Stores Team is one of the critical team within End to End Fulfilment area of Data and Customer Analytics handling Supply chain Stores metrics which enables data driven decision making.

What you'll do:

Design, build, test and deploy cutting edge solutions at scale, impacting multi-billion-dollar business.
Work closely with product owner and technical lead and play a major role in the overall delivery of the assigned project/enhancements.
 Interact with Walmart engineering teams across geographies to leverage expertise and contribute to the tech community.
Provide business insights while leveraging internal tools and systems, databases and industry data.
Drive the success of the implementation by

___

Data preprocessing

In [60]:
# imports
from pyspark.sql import Row
from pyspark.sql.functions import col, lower, regexp_replace
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF

In [37]:
# create spark dataframe
data = [
    Row(id=1, type='resume', text=resume_data),
    Row(id=2, type='job_description',text=job_description)
]
txt_df = spark.createDataFrame(data)

In [42]:
# display df
txt_df.show(truncate=True)

+---+---------------+--------------------+
| id|           type|                text|
+---+---------------+--------------------+
|  1|         resume|ALAN SUSA\nData E...|
|  2|job_description|About the job\nPo...|
+---+---------------+--------------------+



In [44]:
# text cleaning
df_cleaned = (txt_df.withColumn('cleaned_text', lower(regexp_replace(col('text'), '^a-zA-Z0-9\s', ''))))

In [46]:
# display df
df_cleaned.select('text', 'cleaned_text').show(truncate=True)

+--------------------+--------------------+
|                text|        cleaned_text|
+--------------------+--------------------+
|ALAN SUSA\nData E...|alan susa\ndata e...|
|About the job\nPo...|about the job\npo...|
+--------------------+--------------------+



In [52]:
# tokenization
tokenizer = Tokenizer(inputCol='cleaned_text', outputCol='words')
tokenized_df = tokenizer.transform(df_cleaned)

In [53]:
# display df
tokenized_df.select('words').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [55]:
# remove stop words
remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')
filtered_df = remover.transform(tokenized_df)

In [56]:
# display df
filtered_df.select('filtered_words').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [58]:
# convert text to TF-IDF features
cv = CountVectorizer(inputCol='filtered_words', outputCol='raw_features')
cv_model = cv.fit(filtered_df)
vectorized_df = cv_model.transform(filtered_df)

In [59]:
# display df
vectorized_df.select('raw_features').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [64]:
# IDF feature
idf = IDF(inputCol='raw_features', outputCol='tfidf_features')
idf_model = idf.fit(vectorized_df)
tfidf_df = idf_model.transform(vectorized_df)

In [65]:
# display df
tfidf_df.select('tfidf_features').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------